# SpamSentinel — exploratory data analysis (Cresci-2017)

Distributions, class balance and correlations of the 64 account-level features in `data/processed/features.csv`. Run `ml/scripts/build_features.py` first.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from spamsentinel_ml.config import PROCESSED_DIR
from spamsentinel_ml.features import FEATURES, FEATURE_NAMES, FEATURE_INFO
df = pd.read_csv(PROCESSED_DIR / 'features.csv', index_col='user_id')
print(df.shape)
json.loads((PROCESSED_DIR / 'cleaning_report.json').read_text())

## Class balance and account groups

In [ ]:
display(df['label'].map({0: 'genuine', 1: 'spam'}).value_counts())
df['group'].value_counts().plot.barh(title='Accounts per group'); plt.show()

## Missing values

In [ ]:
miss = df[FEATURE_NAMES].isna().mean().sort_values(ascending=False)
miss[miss > 0]

## Feature summary by class
Medians per class and the standardised mean difference (how far apart the classes are, in pooled standard deviations).

In [ ]:
g = df.groupby('label')[FEATURE_NAMES]
med = g.median().T.rename(columns={0: 'genuine_median', 1: 'spam_median'})
m, s = g.mean().T, g.std().T
med['smd'] = (m[1] - m[0]) / np.sqrt((s[0]**2 + s[1]**2) / 2)
med['category'] = [FEATURE_INFO[f].category for f in med.index]
med.sort_values('smd', key=abs, ascending=False).round(3)

## Distributions of the most separating features (log scale where skewed)

In [ ]:
top = med['smd'].abs().sort_values(ascending=False).index[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, f in zip(axes.ravel(), top):
    for lab, name, c in [(0, 'genuine', '#0F766E'), (1, 'spam', '#D50032')]:
        v = df.loc[df.label == lab, f].dropna()
        v = np.log1p(v) if v.max() > 50 else v
        ax.hist(v, bins=40, alpha=0.5, density=True, label=name, color=c)
    ax.set_title(f, fontsize=9)
axes[0, 0].legend(); fig.tight_layout(); plt.show()

## Correlation between features
Highly correlated pairs are redundant — the motivation for PRISMO's feature selection.

In [ ]:
corr = df[FEATURE_NAMES].corr(method='spearman')
fig, ax = plt.subplots(figsize=(12, 10)); im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURE_NAMES))); ax.set_xticklabels(FEATURE_NAMES, rotation=90, fontsize=5)
ax.set_yticks(range(len(FEATURE_NAMES))); ax.set_yticklabels(FEATURE_NAMES, fontsize=5)
fig.colorbar(im); plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
pairs[pairs.abs() > 0.9].sort_values(key=abs, ascending=False)

## Differences between spambot groups
The spam class mixes several bot families with distinct behaviour.

In [ ]:
df.groupby('group')[list(top[:8])].median().T.round(2)